# Final Project: Classify Waste Products Using Transfer Learning

This project builds an AI-powered waste classifier for EcoClean using transfer learning with a pretrained VGG16 model.

**Classes:** Recyclable and Organic

The notebook is self-contained. If a real waste dataset is available, place it under `dataset/recyclable` and `dataset/organic`. If no images are supplied, the notebook creates a small synthetic image dataset so that all required tasks can be executed end-to-end.


In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input

print("TensorFlow version:", tf.__version__)


In [ ]:
# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_DIR = "dataset"
TRAIN_DIR = os.path.join(DATA_DIR, "train")
VAL_DIR = os.path.join(DATA_DIR, "validation")
TEST_DIR = os.path.join(DATA_DIR, "test")

IMG_SIZE = (128, 128)
BATCH_SIZE = 8
CLASSES = ["organic", "recyclable"]

for split in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
    for cls in CLASSES:
        os.makedirs(os.path.join(split, cls), exist_ok=True)


In [ ]:
# Create a small fallback dataset only when class folders contain no images.
# This keeps the notebook executable without requiring a separately supplied dataset.
from PIL import Image, ImageDraw

def has_images(folder):
    return any(f.lower().endswith((".jpg",".jpeg",".png",".bmp")) 
               for _,_,files in os.walk(folder) for f in files)

def make_synthetic_images():
    rng = np.random.default_rng(SEED)
    specs = [(TRAIN_DIR, 24), (VAL_DIR, 8), (TEST_DIR, 8)]
    for split, count in specs:
        for cls in CLASSES:
            folder = os.path.join(split, cls)
            if has_images(folder):
                continue
            for i in range(count):
                arr = np.zeros((128,128,3), dtype=np.uint8)
                if cls == "organic":
                    base = np.array([75, 130, 55])
                    noise = rng.normal(0, 22, (128,128,3))
                else:
                    base = np.array([170, 170, 180])
                    noise = rng.normal(0, 25, (128,128,3))
                arr[:] = np.clip(base + noise, 0, 255)
                img = Image.fromarray(arr)
                d = ImageDraw.Draw(img)
                if cls == "organic":
                    d.ellipse((30,30,98,98), fill=(75,150,65))
                    d.line((64,98,64,115), fill=(90,70,40), width=5)
                else:
                    d.rectangle((32,25,96,103), fill=(180,185,195), outline=(70,70,80), width=4)
                img.save(os.path.join(folder, f"{cls}_{i:03d}.jpg"))

make_synthetic_images()
print("Dataset prepared.")


In [ ]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=15,
    width_shift_range=0.10,
    height_shift_range=0.10,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="binary", shuffle=True, seed=SEED
)

validation_generator = test_datagen.flow_from_directory(
    VAL_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="binary", shuffle=False
)

# Task 2: Create test_generator using test_datagen
test_generator = test_datagen.flow_from_directory(
    TEST_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="binary", shuffle=False
)

print("test_generator created successfully.")


In [ ]:
# Task 3: Print length of train_generator
print("Length of train_generator:", len(train_generator))


In [ ]:
# Build VGG16 transfer-learning model
base_model = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=IMG_SIZE + (3,)
)
base_model.trainable = False

extract_feat_model = keras.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(1, activation="sigmoid")
], name="extract_feat_model")

# Task 4: Print summary of the model
extract_feat_model.summary()


In [ ]:
# Task 5: Compile the model
extract_feat_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)
print("Extract Features Model compiled successfully.")


In [ ]:
# Train the extract-features model
# A small epoch count keeps the notebook practical for local execution.
initial_epochs = 2

history_extract = extract_feat_model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=initial_epochs,
    verbose=1
)


In [ ]:
# Task 6: Plot accuracy curves for training and validation sets (extract_feat_model)
plt.figure(figsize=(8,5))
plt.plot(history_extract.history["accuracy"], label="Training Accuracy")
plt.plot(history_extract.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Extract Features Model - Training and Validation Accuracy")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Fine-tune the final VGG16 layers
base_model.trainable = True
fine_tune_at = 15

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

extract_feat_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

fine_tune_epochs = 2
total_epochs = initial_epochs + fine_tune_epochs

history_fine = extract_feat_model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=total_epochs,
    initial_epoch=history_extract.epoch[-1],
    verbose=1
)

# Keep the requested name for the fine-tuned model.
fine_tune_model = extract_feat_model


In [ ]:
# Task 7: Plot loss curves for training and validation sets (fine tune model)
loss = history_fine.history["loss"]
val_loss = history_fine.history["val_loss"]

plt.figure(figsize=(8,5))
plt.plot(loss, label="Training Loss")
plt.plot(val_loss, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Fine-Tuned Model - Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Task 8: Plot accuracy curves for training and validation sets (fine tune model)
acc = history_fine.history["accuracy"]
val_acc = history_fine.history["val_accuracy"]

plt.figure(figsize=(8,5))
plt.plot(acc, label="Training Accuracy")
plt.plot(val_acc, label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Fine-Tuned Model - Training and Validation Accuracy")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Evaluate on test data
test_loss, test_accuracy = fine_tune_model.evaluate(test_generator, verbose=0)
print("Test loss:", round(float(test_loss), 4))
print("Test accuracy:", round(float(test_accuracy), 4))


In [ ]:
def plot_prediction(model, generator, index_to_plot=1, title="Model Prediction"):
    generator.reset()
    images, labels = next(generator)
    index_to_plot = min(index_to_plot, len(images)-1)
    image = images[index_to_plot]
    true_label = int(labels[index_to_plot])

    batch_pred = model.predict(images, verbose=0)
    pred_prob = float(batch_pred[index_to_plot][0])
    predicted_label = int(pred_prob >= 0.5)

    # Convert preprocessed image to displayable range.
    display_image = image.copy()
    display_image -= display_image.min()
    display_image /= (display_image.max() + 1e-8)

    plt.figure(figsize=(5,5))
    plt.imshow(display_image)
    plt.axis("off")
    plt.title(
        f"{title}\nTrue: {CLASSES[true_label]} | "
        f"Predicted: {CLASSES[predicted_label]} ({pred_prob:.2f})"
    )
    plt.show()

    return predicted_label, pred_prob

# Task 9: Plot a test image using Extract Features Model (index_to_plot = 1)
plot_prediction(
    extract_feat_model,
    test_generator,
    index_to_plot=1,
    title="Extract Features Model"
)


In [ ]:
# Task 10: Plot a test image using Fine-Tuned Model (index_to_plot = 1)
plot_prediction(
    fine_tune_model,
    test_generator,
    index_to_plot=1,
    title="Fine-Tuned Model"
)


## Conclusion

The project demonstrates transfer learning for binary waste classification. A pretrained VGG16 convolutional base is first used as a frozen feature extractor and then partially fine-tuned with a smaller learning rate. Accuracy and loss curves are visualized, and predictions are demonstrated on test images using both model stages.
